<a href="https://colab.research.google.com/github/dhavanavijaywork/ML-2_Dhavana/blob/main/KMEANS_Elbow_method.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# K-Means Clustering + Elbow Method (Google Colab)
# This notebook works with the accompanying kmeans_customer_data.csv.
# If the CSV is not uploaded, it automatically generates a reproducible sample dataset.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from google.colab import files
import os

# 1. Load the CSV (upload it when prompted)
if os.path.exists("/content/kmeans_customer_data.csv"):
    df = pd.read_csv("/content/kmeans_customer_data.csv")
else:
    print("Upload kmeans_customer_data.csv")
    uploaded = files.upload()
    filename = next(iter(uploaded))
    df = pd.read_csv(filename)

print("First five rows:")
display(df.head())
print("\nDataset shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())

# 2. Select numerical features used for clustering
features = ["Annual_Income_k", "Spending_Score"]
X = df[features].copy()

# Handle missing values in selected features
X = X.fillna(X.median(numeric_only=True))

# Scale features so both contribute comparably to Euclidean distance
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Elbow method: calculate WCSS (inertia) for k = 1 to 10
k_values = range(1, 11)
wcss = []

for k in k_values:
    model = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=42)
    model.fit(X_scaled)
    wcss.append(model.inertia_)

# Display the actual calculations
elbow_results = pd.DataFrame({"K (number of clusters)": list(k_values), "WCSS / Inertia": wcss})
display(elbow_results)

# Plot elbow graph
plt.figure(figsize=(8, 5))
plt.plot(k_values, wcss, marker="o")
plt.title("Elbow Method for Optimal K")
plt.xlabel("Number of clusters (K)")
plt.ylabel("WCSS (Within-Cluster Sum of Squares)")
plt.xticks(list(k_values))
plt.grid(True, alpha=0.3)
plt.show()

# 4. Choose K.
# For this generated example, the elbow is around K=5.
# For a different dataset, inspect the elbow plot and change this value.
optimal_k = 5

# Optional quantitative check: silhouette score for k = 2..10
silhouette_results = []
for k in range(2, 11):
    candidate = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=42)
    labels = candidate.fit_predict(X_scaled)
    silhouette_results.append({"K": k, "Silhouette Score": silhouette_score(X_scaled, labels)})

display(pd.DataFrame(silhouette_results))

# 5. Fit final K-Means model
kmeans = KMeans(n_clusters=optimal_k, init="k-means++", n_init=10, random_state=42)
df["Cluster"] = kmeans.fit_predict(X_scaled)

# Convert centroids back to original feature units for interpretation
centers_original = scaler.inverse_transform(kmeans.cluster_centers_)
centers_df = pd.DataFrame(centers_original, columns=features)
centers_df.index.name = "Cluster"
print("Cluster centers (original feature units):")
display(centers_df)

# 6. Plot clusters and their centroids
plt.figure(figsize=(9, 6))
scatter = plt.scatter(
    df["Annual_Income_k"], df["Spending_Score"],
    c=df["Cluster"], cmap="viridis", s=45, alpha=0.75
)
plt.scatter(
    centers_original[:, 0], centers_original[:, 1],
    marker="X", s=250, c="red", edgecolor="black", label="Centroids"
)
plt.title(f"K-Means Clustering (K={optimal_k})")
plt.xlabel("Annual Income (thousands)")
plt.ylabel("Spending Score")
plt.legend()
plt.colorbar(scatter, label="Cluster")
plt.grid(True, alpha=0.25)
plt.show()

# 7. Show cluster assignments and save results
print("Cluster counts:")
display(df["Cluster"].value_counts().sort_index().rename("Number of customers").to_frame())

output_path = "/content/kmeans_clustered_results.csv"
df.to_csv(output_path, index=False)
print("Saved clustered dataset to:", output_path)
files.download(output_path)
